# RAWG API Task — Video Game Data Analysis

Consuming the RAWG API to extract, analyze, and compare video game data using Python.

## Setup — Imports and API Configuration

Importing required libraries and defining the API Key as a local variable.  
**Important:** do not upload your API Key to GitHub.

In [ ]:
import requests
import pandas as pd

# Store your API Key as a local variable — do not share or commit this value
API_KEY = "a52a8e28a079470caa05a06f77a5093e"
BASE_URL = "https://api.rawg.io/api"

def get(endpoint, params=None):
    """Helper to make authenticated GET requests to the RAWG API."""
    if params is None:
        params = {}
    params["key"] = API_KEY
    response = requests.get(f"{BASE_URL}{endpoint}", params=params)
    response.raise_for_status()
    return response.json()

print("Setup complete.")

## Part A — General Exploration

### A1 — How many games does RAWG have registered in total?

We query the `/games` endpoint and read the `count` field from the response, which represents the total number of games in the database.

In [ ]:
data = get("/games")
total_games = data["count"]
print(f"Total number of games registered in RAWG: {total_games:,}")

## Part B — Category Analysis

### B1 — Top 5 highest rated games of all time (by Metacritic)

We query the `/games` endpoint ordering by Metacritic score and display the top 5 results showing name, rating, and metacritic score.

In [ ]:
data_b1 = get("/games", params={"ordering": "-metacritic", "page_size": 5})

top5_metacritic = [
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_b1["results"]
]

df_b1 = pd.DataFrame(top5_metacritic)
df_b1.index += 1
print("Top 5 highest rated games of all time (Metacritic):")
df_b1

### B2 — Top 10 best games available on Steam (store_id=1)

We filter by Steam (`stores=1`) and order by rating to get the 10 best games, showing name, rating, and metacritic score.

In [ ]:
data_b2 = get("/games", params={"stores": 1, "ordering": "-rating", "page_size": 10})

top10_steam = [
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_b2["results"]
]

df_b2 = pd.DataFrame(top10_steam)
df_b2.index += 1
print("Top 10 best games available on Steam:")
df_b2

## Part C — Comparisons

### C1 — Top 5 games on PC (platform_id=4) vs Top 5 on PS5 (platform_id=187)

We query both platforms separately, compare their average ratings, and determine which platform has the highest rated games.

In [ ]:
data_pc = get("/games", params={"platforms": 4, "ordering": "-rating", "page_size": 5})
data_ps5 = get("/games", params={"platforms": 187, "ordering": "-rating", "page_size": 5})

df_pc = pd.DataFrame([
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_pc["results"]
])
df_ps5 = pd.DataFrame([
    {"name": g["name"], "rating": g["rating"], "metacritic": g["metacritic"]}
    for g in data_ps5["results"]
])

df_pc.index += 1
df_ps5.index += 1

avg_pc = df_pc["rating"].mean()
avg_ps5 = df_ps5["rating"].mean()

print("Top 5 games on PC:")
print(df_pc.to_string())
print(f"\nAverage rating on PC: {avg_pc:.2f}")

print("\nTop 5 games on PS5:")
print(df_ps5.to_string())
print(f"\nAverage rating on PS5: {avg_ps5:.2f}")

winner = "PC" if avg_pc > avg_ps5 else "PS5"
print(f"\nConclusion: {winner} has the highest rated games with an average rating of {max(avg_pc, avg_ps5):.2f}.")

### C2 — Comparison table for 3 famous games

We search for 3 iconic games and build a comparison table showing name, rating, metacritic, genres, and platforms.

In [ ]:
famous_games = ["The Witcher 3: Wild Hunt", "Red Dead Redemption 2", "Grand Theft Auto V"]

rows = []
for title in famous_games:
    result = get("/games", params={"search": title, "page_size": 1})
    if result["results"]:
        g = result["results"][0]
        rows.append({
            "name": g["name"],
            "rating": g["rating"],
            "metacritic": g["metacritic"],
            "genres": ", ".join(genre["name"] for genre in g.get("genres", [])),
            "platforms": ", ".join(p["platform"]["name"] for p in g.get("platforms", []))
        })

df_c2 = pd.DataFrame(rows)
df_c2.index += 1
print("Comparison table — 3 famous games:")
df_c2

### C3 — Average rating by genre (4 genres)

We query the top 5 games for each of 4 different genres, calculate the average user rating per genre, and determine which genre produces the best games.

In [ ]:
# Genre IDs: action=4, RPG=5, strategy=10, shooter=2
genres = {"Action": 4, "RPG": 5, "Strategy": 10, "Shooter": 2}

genre_rows = []
for genre_name, genre_id in genres.items():
    data_g = get("/games", params={"genres": genre_id, "ordering": "-rating", "page_size": 5})
    ratings = [g["rating"] for g in data_g["results"]]
    avg = sum(ratings) / len(ratings) if ratings else 0
    genre_rows.append({"genre": genre_name, "avg_rating": round(avg, 2)})

df_c3 = pd.DataFrame(genre_rows).sort_values("avg_rating", ascending=False)
df_c3.index = range(1, len(df_c3) + 1)

print("Average rating by genre (top 5 games each):")
print(df_c3.to_string())
best_genre = df_c3.iloc[0]["genre"]
print(f"\nConclusion: '{best_genre}' produces the best rated games according to users.")

### C4 — Best games from 3 different years

We compare the best games released in 2015, 2018, and 2022, calculating the average Metacritic score per year to determine which year had the highest rated releases.

In [ ]:
years = [2015, 2018, 2022]
year_rows = []

for year in years:
    data_y = get("/games", params={
        "dates": f"{year}-01-01,{year}-12-31",
        "ordering": "-metacritic",
        "page_size": 5
    })
    scores = [g["metacritic"] for g in data_y["results"] if g["metacritic"]]
    avg = sum(scores) / len(scores) if scores else 0
    year_rows.append({"year": year, "avg_metacritic": round(avg, 2)})

df_c4 = pd.DataFrame(year_rows).sort_values("avg_metacritic", ascending=False)
df_c4.index = range(1, len(df_c4) + 1)

print("Average Metacritic score by year (top 5 games each):")
print(df_c4.to_string())
best_year = df_c4.iloc[0]["year"]
print(f"\nConclusion: {best_year} had the highest average Metacritic score ({df_c4.iloc[0]['avg_metacritic']}).")

### C5 — Export top 20 games of all time to CSV

We fetch the top 20 games ordered by Metacritic score and export them to `output/top20_rawg.csv` with columns: `name`, `rating`, `metacritic`, `release_date`, `main_genre`. We then display the first 5 rows.

In [ ]:
data_c5 = get("/games", params={"ordering": "-metacritic", "page_size": 20})

top20 = []
for g in data_c5["results"]:
    main_genre = g["genres"][0]["name"] if g.get("genres") else "N/A"
    top20.append({
        "name": g["name"],
        "rating": g["rating"],
        "metacritic": g["metacritic"],
        "release_date": g["released"],
        "main_genre": main_genre
    })

df_c5 = pd.DataFrame(top20)
df_c5.index += 1

output_path = "output/top20_rawg.csv"
df_c5.to_csv(output_path, index=False)
print(f"CSV exported to {output_path}")
print("\nFirst 5 rows:")
df_c5.head()